# Lecture 4 — Class Exercise
## Scatter & Bubble Charts: Gapminder

> **Push to:** `week04/lecture04_exercise.ipynb`

**Rules:**
1. Colour used **sparingly** — one categorical variable, no rainbow
2. If showing all continents, either use accessible palette OR grey all + highlight one
3. `size_max` set when using bubble size
4. Log scale for GDP per capita
5. Insight title

---


In [1]:
import pandas as pd
import plotly.express as px


# Dataset: Gapminder — GDP, Life Expectancy, Population by Country
# Source: Gapminder Foundation (gapminder.org)

df = px.data.gapminder()
print(f"Loaded: {len(df)} rows")
print(df.head())

Loaded: 1704 rows
       country continent  year  lifeExp       pop   gdpPercap iso_alpha  \
0  Afghanistan      Asia  1952   28.801   8425333  779.445314       AFG   
1  Afghanistan      Asia  1957   30.332   9240934  820.853030       AFG   
2  Afghanistan      Asia  1962   31.997  10267083  853.100710       AFG   
3  Afghanistan      Asia  1967   34.020  11537966  836.197138       AFG   
4  Afghanistan      Asia  1972   36.088  13079460  739.981106       AFG   

   iso_num  
0        4  
1        4  
2        4  
3        4  
4        4  


In [2]:
# explore

print(df.info())
print("Years:", sorted(df['year'].unique()))
print("Continents:", df['continent'].unique())
print(df.describe().round(1))


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1704 entries, 0 to 1703
Data columns (total 8 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   country    1704 non-null   object 
 1   continent  1704 non-null   object 
 2   year       1704 non-null   int64  
 3   lifeExp    1704 non-null   float64
 4   pop        1704 non-null   int64  
 5   gdpPercap  1704 non-null   float64
 6   iso_alpha  1704 non-null   object 
 7   iso_num    1704 non-null   int64  
dtypes: float64(2), int64(3), object(3)
memory usage: 106.6+ KB
None
Years: [1952, 1957, 1962, 1967, 1972, 1977, 1982, 1987, 1992, 1997, 2002, 2007]
Continents: ['Asia' 'Europe' 'Africa' 'Americas' 'Oceania']
         year  lifeExp           pop  gdpPercap  iso_num
count  1704.0   1704.0  1.704000e+03     1704.0   1704.0
mean   1979.5     59.5  2.960121e+07     7215.3    425.9
std      17.3     12.9  1.061579e+08     9857.5    248.3
min    1952.0     23.6  6.001100e+04      241.2      4.0


## Task 1 — Scatter: life expectancy change over time

**What to build:** A scatter showing **GDP per capita vs life expectancy** for **two years** (2002 and 2007) to show how both moved — use **colour for year** (just 2 colours), **one continent only**.

Choose any continent except Africa (that was the example). Highlight the change direction.

> 💡 Filter: `df.loc[df['continent'] == 'YOUR_CHOICE']` then filter years


In [4]:

import plotly.graph_objects as go

# Choose one continent and two years
continent_choice = 'Asia'
years = [2002, 2007]

# Filter the data
task1 = df.loc[
    (df['continent'] == continent_choice) &
    (df['year'].isin(years))
].copy()

# Create the figure
fig1 = go.Figure()

# Connect each country's 2002 and 2007 observations
for country, group in task1.groupby('country'):
    group = group.sort_values('year')

    if len(group) == 2:
        fig1.add_trace(go.Scatter(
            x=group['gdpPercap'],
            y=group['lifeExp'],
            mode='lines',
            line=dict(color='lightgrey', width=1),
            showlegend=False,
            hoverinfo='skip'
        ))

# Add the two years using exactly two colours
for year, color in [
    (2002, '#2878B5'),
    (2007, '#E07A25')
]:
    d = task1.loc[task1['year'] == year]

    fig1.add_trace(go.Scatter(
        x=d['gdpPercap'],
        y=d['lifeExp'],
        mode='markers',
        name=str(year),
        marker=dict(color=color, size=9),
        text=d['country'],
        hovertemplate=(
            '<b>%{text}</b><br>'
            'GDP per capita: $%{x:,.0f}<br>'
            'Life expectancy: %{y:.1f} years'
            '<extra></extra>'
        )
    ))

# Add the title and labels
fig1.update_layout(
    title='Asia, 2002–2007: GDP per capita and life expectancy changes',
    xaxis_title='GDP per capita (US dollars)',
    yaxis_title='Life expectancy (years)',
    template='plotly_white',
    legend_title='Year'
)

fig1.show()


## Task 2 — Bubble chart: tell a story

**What to build:** A bubble chart (full 2007 dataset, all countries) where:
- x = GDP per capita (log scale)
- y = life expectancy
- size = population
- colour = ONE continent highlighted (your choice), all others grey
- At least one annotation explaining the highlighted group's story

> This is the grey-and-highlight technique applied to a bubble chart.


In [6]:

# Task 2 — Bubble chart: highlight Asia against all other continents

year_choice = 2007
highlight_continent = 'Asia'

# Keep all countries in 2007
task2 = df.loc[df['year'] == year_choice].copy()

# Create a group for highlighting
task2['group'] = task2['continent'].apply(
    lambda continent: 'Asia' if continent == highlight_continent
    else 'Other continents'
)

# Grey for other continents and orange for Asia
fig2 = px.scatter(
    task2,
    x='gdpPercap',
    y='lifeExp',
    size='pop',
    size_max=42,
    color='group',
    color_discrete_map={
        'Asia': '#D95F02',
        'Other continents': '#B8B8B8'
    },
    hover_name='country',
    hover_data={
        'continent': True,
        'pop': ':,',
        'gdpPercap': ':,.0f',
        'lifeExp': ':.1f',
        'group': False
    },
    log_x=True,
    title='2007: Asia spans wide income and life expectancy differences',
    labels={
        'gdpPercap': 'GDP per capita (US dollars, log scale)',
        'lifeExp': 'Life expectancy (years)',
        'group': 'Continent group',
        'pop': 'Population'
    }
)

# Add an explanatory annotation
fig2.add_annotation(
    x=1800,
    y=72,
    text='Asia is diverse: countries span a wide range<br>'
         'of incomes and life expectancies.',
    showarrow=True,
    arrowhead=2,
    ax=85,
    ay=-45
)

fig2.update_layout(template='plotly_white')

fig2.show()
